# Firms cleaning

Builds three lists of firms for the `firms` table from the client's vendor export,
their cage maintenance table, and our internal cages table:

1. **Linked vendor-manufacturers**: a vendor row followed by the manufacturers it represents, grouped by `link_id`
2. **Orphan vendors**: vendors with no manufacturer linked to them (they represent themselves)
3. **Orphan manufacturers**: manufacturers with no vendor linked to them

Every list has `company_name, cage, email, phone, contact_name` (all nullable).
The linked list also has `link_id` and `role` (`vendor` or `manufacturer`).

Rows that get dropped along the way are written to `output/rejected.csv` with a reason,
so nothing disappears silently.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

DATA = "data"
OUT = "output"
VENDOR_FILE = f"{DATA}/Sales Patriot Project Vendor File_ANONYMIZED.csv"
MAINT_FILE = f"{DATA}/Cage Maintenance_ANONYMIZED.csv"
CAGES_FILE = f"{DATA}/cages_ANONYMIZED.csv"

FIRM_COLUMNS = ["company_name", "cage", "email", "phone", "contact_name"]
LINKED_COLUMNS = FIRM_COLUMNS + ["link_id", "role"]

## 1. Load

Everything is read as text so codes like `000123` keep their leading zeros.
The cages table is about 590 MB and 4 million rows, so only the three columns we need are loaded.

In [2]:
vendors_raw = pd.read_csv(VENDOR_FILE, dtype=str, low_memory=False)
maint_raw = pd.read_csv(MAINT_FILE, dtype=str)
cages_raw = pd.read_csv(CAGES_FILE, dtype=str, usecols=["id", "company", "status"])

for name, df in [("vendors", vendors_raw), ("cage maintenance", maint_raw), ("cages", cages_raw)]:
    print(f"{name:17} {df.shape[0]:>9,} rows  {df.shape[1]:>4} columns")

vendors              18,318 rows   139 columns
cage maintenance     20,385 rows     5 columns
cages             4,005,840 rows     3 columns


## 2. Inspect the raw files

Before cleaning anything, look at what each file actually contains and how they connect.

### Vendor file
139 columns, most of them almost empty. The ones that matter for a firm are the id, name,
phone, contact and email. There is **no cage code column**, so a vendor's cage has to be found
by matching its name against the cages table.

In [3]:
filled = vendors_raw.notna().sum().sort_values(ascending=False)
print("columns more than half filled:", len(filled[filled > len(vendors_raw) / 2]), "of", len(filled))
print("any column mentioning cage:", [c for c in vendors_raw.columns if "cage" in c.lower()])

vendor_cols = ["VendorId", "VendorName", "Phone", "Contact", "EmailAddress"]
vendors_raw[vendor_cols].head(10)

columns more than half filled: 29 of 139
any column mentioning cage: []


,VendorId,VendorName,Phone,Contact,EmailAddress
0,008223,DO NOT USE,455-5597,ARTHUR SIMPLINA,NaN
1,008224,YORK INDUSTRIES,800-964-7035,NaN,NaN
2,008225,BILL RAPP SUBARU,626-239-8066,Jaimie & Wade Scarpitti,NaN
3,008226,EPICOR SOFTWARE CORP.,602-595-6430,Michael Duncan,sales@partexmarker.com
4,008227,ONONDAGA COUNTY CLERK,866-802-6383,NaN,NaN
5,008228,J & J INDUSTRIES INC.,704-509-4328,NaN,NaN
6,008229,PREFERRED POWER IND.,559-688-2911,sales,NaN
7,008230,DAYTON WALTHER,800-521-4888,SCOTT HINDLEY,NaN
8,008231,AUSTIN HARDWARE,614-228-5781,Brian Wider,NaN
9,008232,"JAGUAR INDUSTRIES, INC.",800-433-0038,JAMIE BLAKE,NaN


In [4]:
vendors_raw[vendor_cols].notna().sum().to_frame("non-null")

,non-null
VendorId,18318
VendorName,18292
Phone,17032
Contact,14063
EmailAddress,8363


In [5]:
# ids are zero-padded text, a few are not numbers at all
print(vendors_raw["VendorId"].sample(5, random_state=1).tolist())
print(vendors_raw.loc[pd.to_numeric(vendors_raw["VendorId"], errors="coerce").isna(), ["VendorId", "VendorName"]])

['111519', '102909', '108696', '111747', '106973']
      VendorId           VendorName
622     00TEST         PRELUDE TEST
1912    00000M                  NaN
2322    000PSI  PRELUDE TEST VENDOR
18317  REPORTS                  NaN


In [6]:
# a first look at names that say the vendor should not be used, and exact duplicate names.
# the cleaning below catches more: redirects like "USE 104806", and duplicates after normalising names
junk =vendors_raw["VendorName"].str.contains("DO NOT USE|INACTIVE", case=False, na=False)
print("do not use / inactive:", junk.sum())
print("duplicate names:", vendors_raw["VendorName"].str.upper().str.strip().duplicated().sum())
vendors_raw.loc[junk, "VendorName"].head(8).tolist()

do not use / inactive: 50
duplicate names: 285


['DO NOT USE',
 'DO NOT USE. SEE V#101020',
 'DO NOT USE DUP VENDOR SEE ',
 'INACTIVE UNITED PARCEL SERVICE',
 'INACTIVE UNITED PARCEL SERVICE',
 'FLEETPRIDE  *DO NOT USE*',
 'DO NOT USE SEE 3778',
 'INACTIVE UPS SUPPLY CHAIN']

### Cage maintenance
One row per manufacturer cage code. `VendorCode` links the manufacturer to a vendor, and
`Description` is a free-text note: mostly emails, sometimes fax numbers or remarks.

In [7]:
maint_raw.head(10)

,CageCode,Description,VendorCode,PreludeKey,ExportDate
0,NaN,SALES@MILEHIMACHINE.COM,2538.0,001!,2024-07-09 16:15:44.000
1,/tai,NaN,NaN,001! /tai,2024-07-09 16:15:44.000
2,03038,NaN,NaN,001! 03038,2024-07-09 16:15:44.000
3,07690,NaN,NaN,001! 07690,2024-07-09 16:15:44.000
4,0ACP0,DIST,NaN,001! 0ACP0,2024-07-09 16:15:44.000
5,0KXH9,bhurley@spi-co.com,NaN,001! 0KXH9,2024-07-09 16:15:44.000
6,18863,NaN,NaN,001! 18863,2024-07-09 16:15:44.000
7,19522,ESAB DIST,NaN,001! 19522,2024-07-09 16:15:44.000
8,1CRL3,dcunningham@marshbellofram.com,NaN,001! 1CRL3,2024-07-09 16:15:44.000
9,1M7N0,cmccoy@krayden.com,NaN,001! 1M7N0,2024-07-09 16:15:44.000


In [8]:
maint_raw.notna().sum().to_frame("non-null")

,non-null
CageCode,20384
Description,16784
VendorCode,11982
PreludeKey,20385
ExportDate,20385


In [9]:
# cage codes should be 5 characters; some are not
maint_raw["CageCode"].str.strip().str.len().dropna().astype(int).value_counts().sort_index()

CageCode
1        4
2        3
3        8
4      137
5    20108
6      124
Name: count, dtype: int64

In [10]:
desc = maint_raw["Description"].dropna()
print("descriptions with an email:", desc.str.contains("@").sum(), "of", len(desc))
desc[~desc.str.contains("@")].sample(10, random_state=4).tolist()

descriptions with an email: 12906 of 16784


['CURTIS (F) 440-446-0555',
 'ARMOR (F)904-807-5328',
 'LIPHATECH (F) 414-247-8166',
 'LOUISVLLE LADDER ',
 'DO NOT USE ',
 'KWIK FILE (F) 763-572-0168',
 'PA LEONE&SONS(F)3154721824',
 'WILL NOT QUOTE W/OUT SERIAL#',
 'NOSHOK',
 'V# 6997  ']

`VendorCode` is stored as a float (`2538.0`) while `VendorId` is zero-padded text (`002538`),
so the two only match once both are converted to numbers.

In [11]:
vendor_ids = pd.to_numeric(vendors_raw["VendorId"], errors="coerce")
vendor_codes = pd.to_numeric(maint_raw["VendorCode"], errors="coerce").dropna()
print("linked manufacturer rows:", len(vendor_codes))
print("  as text, found in vendor file:", maint_raw["VendorCode"].isin(vendors_raw["VendorId"]).sum())
print("  as numbers, found in vendor file:", vendor_codes.isin(vendor_ids).sum())

linked manufacturer rows: 11982
  as text, found in vendor file: 0
  as numbers, found in vendor file: 11973


In [12]:
# the most linked vendors; 5 and 2 are in the vendor file with a blank name (so is vendor 0)
top = vendor_codes.astype(int).value_counts().head(6).rename_axis("vendor_code").reset_index(name="manufacturers")
names = vendors_raw.assign(vendor_code=vendor_ids.astype("Int64")).dropna(subset=["vendor_code"]).set_index("vendor_code")["VendorName"]
top["VendorName"] = top["vendor_code"].map(names)
top

,vendor_code,manufacturers,VendorName
0,5,3405,NaN
1,3197,250,MASLINE ELECTRONICS INC.
2,100078,124,PIONEER INDUSTRIES INC.
3,2,64,NaN
4,109,48,HAUN WELDING SUPPLY
5,5974,44,"WESCO DISTRIBUTION, INC."


### Cages table
Our reference table: one row per cage code with the official company name.

In [13]:
print("cage ids unique:", cages_raw["id"].is_unique)
print("cage id lengths:", cages_raw["id"].str.len().value_counts().to_dict())
print("missing company names:", cages_raw["company"].isna().sum())
cages_raw["status"].value_counts().to_frame()

cage ids unique: True
cage id lengths: {5: 4005840}
missing company names: 301


,count
status,
A,2837931
H,534696
N,321714
P,135274
R,113351
F,28645
C,18047
Y,11577
E,3544


In [14]:
maint_cages = maint_raw["CageCode"].str.strip().str.upper()
print("maintenance cage codes found in cages table:", maint_cages.isin(cages_raw["id"]).sum(), "of", maint_cages.notna().sum())

maintenance cage codes found in cages table: 20007 of 20384


### What this means for the cleaning

- Manufacturer names come from the cages table, looked up by cage code.
- Vendors need their cage found by name, so names have to be normalised before matching.
- Vendor ids and vendor codes are compared as numbers.
- Vendors 0, 2 and 5 have no name, so the manufacturers pointing at them have no real parent and are treated as orphans.
- Emails are pulled out of `Description`; fax numbers and notes there are ignored.

## 3. Cleaning helpers

`name_key` is what makes name matching work: `"Acme Co."`, `"ACME, INC"` and `"The Acme Company"`
all become `ACME`. It is only used for comparing names; the output keeps the original name.

In [15]:
CAGE_PATTERN = r"^[A-Z0-9]{5}$"
EMAIL_PATTERN = r"([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})"
SUFFIXES = r"\b(?:INC|INCORPORATED|CORP|CORPORATION|CO|COMPANY|LLC|LTD|LIMITED|THE)\b"
# vendor names like "DO NOT USE", "INACTIVE UPS", "GRAYBAR - USE 001270", "*SEE V#3257"
DO_NOT_USE = r"DO NOT USE|\bINACTIVE\b|PRELUDE TEST|\b(?:USE|SEE)\s*(?:V#|#|VC)?\s*\d"
# the vendor id in a redirect like "GRAYBAR - USE 001270"
REDIRECT = r"(?i)\b(?:USE|SEE)\s*(?:V#|#|VC)?\s*(\d+)"
# firms that no longer exist, e.g. "American Air Cond. Co *CLOSED*", "SMEDEGAARD T A/S - OBSOLETE 2017"
CLOSED = r"OBSOLETE|DUPE CODE|FACILITY CLOSED|\bCLOSED\b"
# notes the cages table keeps inside names, e.g. "SWAGELOK CO DBA USE CAGE CODE 3T9T9 FOR CATALOGING"
NAME_NOTES = r"(?:\s+DBA)?\s+(?:FOR PROCUREMENT\s+)?USE CAGE CODE \w+(?:\s+FOR CATALOGING)?\.?|\s+PO BOX\b.*$"
# contacts that are a department rather than a person
NOT_A_PERSON = r"(?i)sales( dept\.?| department)?|customer service|parts|gov't sales|\**see comments\**"


def name_key(names):
    s = names.fillna("").str.upper().str.replace("&", " AND ", regex=False)
    s = s.str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(SUFFIXES, " ", regex=True)
    return s.str.split().str.join(" ").replace("", pd.NA)


def clean_text(s):
    return s.str.strip().str.replace(r"\s+", " ", regex=True).replace("", pd.NA)


def clean_name(s):
    s = clean_text(s.str.replace(NAME_NOTES, "", regex=True, case=False))
    return s.str.replace(r"[\s;,]+$", "", regex=True).replace("", pd.NA)


def clean_contact(s):
    # drop extensions ("Mike Musto X213"), then anything that isn't a person's name
    s = clean_text(s.str.replace(r"\s*\b(?:x|ext)\.?\s*\d+\s*$", "", regex=True, case=False))
    not_a_name = s.str.fullmatch(NOT_A_PERSON) | s.str.contains(r"@|\d")
    return s.mask(not_a_name.fillna(False))


def clean_email(s):
    return s.str.extract(EMAIL_PATTERN, expand=False).str.lower()


def clean_phone(s):
    # keep 10-digit US numbers as 555-555-5555, drop anything else
    digits = s.fillna("").str.replace(r"\D", "", regex=True)
    digits = digits.where(~((digits.str.len() == 11) & digits.str.startswith("1")), digits.str[1:])
    formatted = digits.str[:3] + "-" + digits.str[3:6] + "-" + digits.str[6:]
    return formatted.where(digits.str.len() == 10)


rejected = []


def reject(df, source, id_col, name_col, reason):
    rejected.append(pd.DataFrame({
        "source": source,
        "source_id": df[id_col].values,
        "company_name": df[name_col].values,
        "reason": reason,
    }))

In [16]:
print(name_key(pd.Series(["Acme Co.", "ACME, INC", "The Acme Company", "B & B Hose"])).tolist())
print(clean_name(pd.Series(["SWAGELOK CO DBA USE CAGE CODE 3T9T9 FOR CATALOGING", "VITRAMON INC PO BOX 544 BRIDGEPORT CT 06601", "POWER PARAGON INC;"])).tolist())
print(clean_contact(pd.Series(["Mike Musto X213", "SALES DEPT.", "FAX 215-236-1390", "BONNIE DIERINGER"])).tolist())
print(clean_phone(pd.Series(["800-964-7035", "1 (626) 239-8066", "455-5597", None])).tolist())

['ACME', 'ACME', 'ACME', 'B AND B HOSE']
['SWAGELOK CO', 'VITRAMON INC', 'POWER PARAGON INC']
['Mike Musto', nan, nan, 'BONNIE DIERINGER']
['800-964-7035', '626-239-8066', nan, nan]


## 4. Cages lookup

Slim version of the cages table: cage, company name, status, and the name key for matching.

In [17]:
cages = cages_raw.rename(columns={"id": "cage", "company": "company_name"})
cages["cage"] = cages["cage"].str.strip().str.upper()
cages["company_name"] = clean_name(cages["company_name"])
cages = cages.dropna(subset=["company_name"])
cages["key"] = name_key(cages["company_name"])
print(f"{len(cages):,} cages with a company name")
cages.head()

4,005,539 cages with a company name


,cage,status,company_name,key
0,5NKC1,A,CITY OF WARSAW MUNICIPALITY,CITY OF WARSAW MUNICIPALITY
1,5NKC2,A,CIESPACE CORPORATION,CIESPACE
2,5NKC3,A,"SAMPSON-BLADEN OIL COMPANY, INCORPORATED DBA H...",SAMPSON BLADEN OIL DBA HANDEE HUGO S DIV LUBRI...
3,5NKC4,A,LM-JB INVESTMENTS LLC DBA WASH DAY LAUNDRY,LM JB INVESTMENTS DBA WASH DAY LAUNDRY
4,5NKC5,A,TOWN OF HAW RIVER DIV HAW RIVER TOWN HALL,TOWN OF HAW RIVER DIV HAW RIVER TOWN HALL


## 5. Manufacturers (cage maintenance)

- cage codes are trimmed and upper-cased; anything that is not 5 letters/digits is rejected
- the company name comes from the cages table; cages we can't find are rejected, and so are
  firms the cages table marks as closed, obsolete or a duplicate code
- the vendor code becomes a number so it matches the vendor file
- the email is pulled out of the description
- a cage that appears twice keeps the row with a vendor code, then the row with an email

In [18]:
maint = maint_raw.copy()
maint["cage"] = maint["CageCode"].str.strip().str.upper()
maint["vendor_id"] = pd.to_numeric(maint["VendorCode"], errors="coerce").astype("Int64")
maint["email"] = clean_email(maint["Description"])

bad_cage = ~maint["cage"].fillna("").str.match(CAGE_PATTERN)
reject(maint[bad_cage], "cage_maintenance", "CageCode", "Description", "invalid cage code")
maint = maint[~bad_cage]

maint = maint.merge(cages[["cage", "company_name", "key"]], on="cage", how="left")
missing = maint["company_name"].isna()
reject(maint[missing], "cage_maintenance", "CageCode", "Description", "cage not in cages table")
maint = maint[~missing]

closed = maint["company_name"].str.contains(CLOSED, case=False)
reject(maint[closed], "cage_maintenance", "CageCode", "company_name", "marked closed / obsolete")
maint = maint[~closed]

maint = maint.sort_values(["vendor_id", "email"], na_position="last")
dupe_cages = maint["cage"].duplicated()
reject(maint[dupe_cages], "cage_maintenance", "CageCode", "company_name", "duplicate cage")
maint = maint[~dupe_cages]

print(f"invalid cage codes: {bad_cage.sum()}, not in cages table: {missing.sum()}, "
      f"closed / obsolete: {closed.sum()}, duplicates: {dupe_cages.sum()}")
print(f"{len(maint):,} manufacturers, {maint['vendor_id'].notna().sum():,} linked to a vendor code")
maint[["cage", "company_name", "email", "vendor_id"]].head()

invalid cage codes: 280, not in cages table: 108, closed / obsolete: 4, duplicates: 16
19,977 manufacturers, 11,915 linked to a vendor code


,cage,company_name,email,vendor_id
5851,1H507,ORTON INDUSTRIES INC,amorgan@fpspower.com,0
3604,0T082,POWER DYNAMICS INNOVATIONS LLC,es.support@allenvanguard.com,0
12924,59154,WILSON COMPANY DBA WILSON CO,ethan.mayeux@assaabloy.com,0
3471,0RKK0,CIW INC,margaret.killeen@minarikcorp.com,0
17826,93812,"METALTEK INTERNATIONAL, INC. DBA METALTEK INTE...",rbright@interpower.com,0


## 6. Vendors

Contact fields are cleaned, then rows that can't become a firm are rejected:
ids that aren't numbers (test/system rows), missing names, names marked do-not-use or inactive,
and firms marked closed.

A rejected vendor that points somewhere else ("GRAYBAR - USE 001270") passes its manufacturers
on to that vendor, so the link isn't lost.

In [19]:
vendors = vendors_raw.copy()
vendors["vendor_id"] = pd.to_numeric(vendors["VendorId"], errors="coerce").astype("Int64")
vendors["company_name"] = clean_name(vendors["VendorName"])
vendors["email"] = clean_email(vendors["EmailAddress"])
vendors["phone"] = clean_phone(vendors["Phone"])
vendors["contact_name"] = clean_contact(vendors["Contact"])
vendors["key"] = name_key(vendors["company_name"])

vendors["reason"] = pd.NA
checks = [
    (vendors["vendor_id"].isna(), "non-numeric vendor id"),
    (vendors["company_name"].isna(), "missing vendor name"),
    (vendors["company_name"].str.contains(DO_NOT_USE, case=False, na=False), "marked do-not-use / inactive"),
    (vendors["company_name"].str.contains(CLOSED, case=False, na=False), "marked closed / obsolete"),
]
for mask, reason in checks:
    vendors.loc[mask & vendors["reason"].isna(), "reason"] = reason

bad = vendors["reason"].notna()
for reason, group in vendors[bad].groupby("reason"):
    reject(group, "vendor_file", "VendorId", "VendorName", reason)
print(vendors.loc[bad, "reason"].value_counts().to_string())

target = vendors.loc[bad, "company_name"].str.extract(REDIRECT, expand=False)
moved = vendors[bad].assign(target=pd.to_numeric(target, errors="coerce").astype("Int64"))
moved = moved[moved["vendor_id"].notna() & moved["target"].isin(vendors.loc[~bad, "vendor_id"])]
redirect = dict(zip(moved["vendor_id"], moved["target"]))
print(f"redirects: {len(redirect)} vendors, {maint['vendor_id'].isin(list(redirect)).sum()} manufacturers moved")
maint["vendor_id"] = maint["vendor_id"].replace(redirect)

vendors = vendors[~bad].drop(columns="reason")
print(f"{len(vendors):,} vendors left")

reason
marked do-not-use / inactive    104
missing vendor name              24
non-numeric vendor id             4
marked closed / obsolete          3
redirects: 62 vendors, 40 manufacturers moved
18,183 vendors left


## 7. Duplicate vendors

Two vendor rows with the same name key are the same firm. Keep the one with the most linked
manufacturers, then the one with the most contact details filled in. Manufacturers that pointed
at a dropped row are moved to the row we kept, so no link is lost.

In [20]:
child_counts = maint["vendor_id"].value_counts()
vendors["children"] = vendors["vendor_id"].map(child_counts).fillna(0)
vendors["filled"] = vendors[["email", "phone", "contact_name"]].notna().sum(axis=1)
vendors = vendors.sort_values(["children", "filled"], ascending=False)

kept_id = vendors.groupby("key")["vendor_id"].transform("first")
dupe_vendors = vendors["vendor_id"] != kept_id
reject(vendors[dupe_vendors], "vendor_file", "VendorId", "VendorName", "duplicate vendor name")
remap = dict(zip(vendors.loc[dupe_vendors, "vendor_id"], kept_id[dupe_vendors]))
maint["vendor_id"] = maint["vendor_id"].replace(remap)
vendors = vendors[~dupe_vendors]
print(f"dropped {dupe_vendors.sum()} duplicate vendors, {len(vendors):,} left")

dropped 500 duplicate vendors, 17,683 left


## 8. Find each vendor's cage

Only confident matches; otherwise the cage stays empty.

1. **Its own manufacturer.** If one of the vendor's linked manufacturers has the same name key,
   that manufacturer is the vendor itself and its cage is the vendor's cage.
2. **Exact name match.** The vendor's name key matches exactly one firm in the cages table,
   or exactly one *active* firm when there are several.
3. **An unlinked manufacturer with the same name.** When the cages table has several firms with
   the vendor's name but the client's own manufacturer list has exactly one, use that one
   (by name key, or by exact name when several share the key).

In [21]:
own = maint.dropna(subset=["vendor_id"]).merge(vendors[["vendor_id", "key"]], on=["vendor_id", "key"])
own = own.drop_duplicates("vendor_id", keep=False)
vendors["cage"] = vendors["vendor_id"].map(own.set_index("vendor_id")["cage"])
from_own = vendors["cage"].notna().sum()

candidates = cages[cages["key"].isin(vendors["key"])]
active = candidates[candidates["status"] == "A"]
single = candidates[candidates["key"].map(candidates["key"].value_counts()) == 1]
single_active = active[active["key"].map(active["key"].value_counts()) == 1]
lookup = pd.concat([single, single_active]).drop_duplicates("key").set_index("key")["cage"]
vendors["cage"] = vendors["cage"].fillna(vendors["key"].map(lookup))
from_cages = vendors["cage"].notna().sum() - from_own

unlinked = maint[~maint["vendor_id"].isin(vendors["vendor_id"]) & ~maint["cage"].isin(vendors["cage"].dropna())]
namesakes = unlinked.drop_duplicates("key", keep=False).set_index("key")["cage"]
vendors["cage"] = vendors["cage"].fillna(vendors["key"].map(namesakes))
# when several share the name key, an exact name match still picks one
exact = unlinked.assign(upper=unlinked["company_name"].str.upper()).drop_duplicates("upper", keep=False)
exact = exact[~exact["cage"].isin(vendors["cage"].dropna())].set_index("upper")["cage"]
vendors["cage"] = vendors["cage"].fillna(vendors["company_name"].str.upper().map(exact))

print(f"cage from own manufacturer:      {from_own:,}")
print(f"cage from cages table name:      {from_cages:,}")
print(f"cage from unlinked manufacturer: {vendors['cage'].notna().sum() - from_own - from_cages:,}")
print(f"no confident match:              {vendors['cage'].isna().sum():,}")

cage from own manufacturer:      1,667
cage from cages table name:      4,638
cage from unlinked manufacturer: 246
no confident match:              11,132


In [22]:
# spot check: vendor name next to the name the cages table has for the matched cage
check = vendors.dropna(subset=["cage"]).merge(cages[["cage", "company_name"]], on="cage", suffixes=("", "_in_cages"))
check[["company_name", "company_name_in_cages", "cage"]].sample(10, random_state=7)

,company_name,company_name_in_cages,cage
892,"STEPP MANUFACTURING CO., INC.",STEPP MANUFACTURING CO INC,06447
4680,"SYNTAK INDUSTRIES, INC.","SYNTAK INDUSTRIES CO., INC.",09UP2
351,CORTLAND COMPANY,"CORTLAND COMPANY, INC.",1V795
2495,ITT CANNON RF PRODUCTS,ITT CANNON RF PRODUCTS,98291
5948,GREY HOUSE PUBLISHING,"GREY HOUSE PUBLISHING, INC",3XET2
6043,"THERM PROCESSES, INC.",THERM PROCESSES INC,63427
4455,HYDRAULIC SUPPLY Company,HYDRAULIC SUPPLY CO,0B1P9
6049,IMPACT BUSINESS MACHINES INC.,IMPACT BUSINESS MACHINES,1AT58
4376,MINNESOTA RUBBER,MINNESOTA RUBBER CO,07322
5129,AIRCRAFT METAL PRODUCTS,AIRCRAFT METAL PRODUCTS CORP.,50885


## 9. Build the three lists

A vendor is **linked** when at least one manufacturer points at it. A manufacturer whose cage
is the vendor's own cage is the vendor itself, not a child, so it is left out of the group
(its email is kept if the vendor has none). A vendor whose only "manufacturer" was itself
ends up as an orphan vendor (it represents itself).

Manufacturers pointing at a vendor we rejected or never had (like the nameless vendors 0, 2 and 5)
have no real parent, so they become orphan manufacturers.

In [23]:
children = maint[maint["vendor_id"].isin(vendors["vendor_id"])]
parent_cage = children["vendor_id"].map(vendors.set_index("vendor_id")["cage"])
is_self = children["cage"] == parent_cage.fillna("")
self_rows = children[is_self]
children = children[~is_self]
linked_ids = set(children["vendor_id"])

vendors["email"] = vendors["email"].fillna(vendors["vendor_id"].map(self_rows.set_index("vendor_id")["email"]))
reject(self_rows, "cage_maintenance", "CageCode", "company_name", "same firm as its vendor")

parents = vendors[vendors["vendor_id"].isin(linked_ids)].assign(role="vendor")
children = children.assign(role="manufacturer")

linked = pd.concat([parents, children], ignore_index=True)
linked["link_id"] = linked["vendor_id"].astype(str)
linked["order"] = (linked["role"] == "manufacturer").astype(int)
linked = linked.sort_values(["vendor_id", "order", "company_name", "cage"])[LINKED_COLUMNS]

orphan_vendors = vendors[~vendors["vendor_id"].isin(linked_ids)]
orphan_manufacturers = maint[~maint["vendor_id"].isin(linked_ids) & ~maint["cage"].isin(self_rows["cage"])]

print(f"linked: {linked['link_id'].nunique():,} vendors with {len(children):,} manufacturers")
print(f"orphan vendors: {len(orphan_vendors):,}, orphan manufacturers: {len(orphan_manufacturers):,}")

linked: 3,641 vendors with 6,741 manufacturers
orphan vendors: 14,042, orphan manufacturers: 11,558


## 10. Remove duplicates across the lists

A cage code identifies one firm, so **the same cage means the same firm**. Two rows with the same
name but different cages are kept as two firms (often separate sites or divisions).

- Linked firms win: an orphan vendor is dropped when its cage is in the linked list, or when it has
  no cage and its name key is in the linked list.
- Between the orphan lists the vendor wins, since it carries the contact details: an orphan
  manufacturer is dropped when its cage is already listed as a linked firm or an orphan vendor.

In [24]:
linked_cages = set(linked["cage"].dropna())
linked_keys = set(name_key(linked["company_name"]).dropna())

in_linked = orphan_vendors["cage"].isin(linked_cages) | (
    orphan_vendors["cage"].isna() & orphan_vendors["key"].isin(linked_keys)
)
reject(orphan_vendors[in_linked], "vendor_file", "VendorId", "company_name", "orphan vendor already in linked list")
orphan_vendors = orphan_vendors[~in_linked]
print(f"orphan vendors dropped: {in_linked.sum()}")

listed = linked_cages | set(orphan_vendors["cage"].dropna())
already_listed = orphan_manufacturers["cage"].isin(listed)
reject(orphan_manufacturers[already_listed], "cage_maintenance", "cage", "company_name", "orphan manufacturer already listed")
orphan_manufacturers = orphan_manufacturers[~already_listed]
print(f"orphan manufacturers dropped: {already_listed.sum()}")

orphan_vendors = orphan_vendors.sort_values(["company_name", "cage"])[FIRM_COLUMNS]
orphan_manufacturers = orphan_manufacturers.sort_values(["company_name", "cage"]).reindex(columns=FIRM_COLUMNS)

orphan vendors dropped: 231
orphan manufacturers dropped: 755


## 11. Results

In [25]:
summary = pd.DataFrame({
    name: {
        "rows": len(df),
        "cage filled %": round(df["cage"].notna().mean() * 100, 1),
        "email filled %": round(df["email"].notna().mean() * 100, 1),
        "phone filled %": round(df["phone"].notna().mean() * 100, 1),
    }
    for name, df in [
        ("linked vendor-manufacturers", linked),
        ("orphan vendors", orphan_vendors),
        ("orphan manufacturers", orphan_manufacturers),
    ]
}).T.astype({"rows": int})
summary

,rows,cage filled %,email filled %,phone filled %
linked vendor-manufacturers,10382,77.0,59.3,32.3
orphan vendors,13811,37.3,42.4,85.7
orphan manufacturers,10803,100.0,65.0,0.0


In [26]:
# one linked group as an example
linked[linked["link_id"] == linked["link_id"].value_counts().index[10]]

,company_name,cage,email,phone,contact_name,link_id,role
12,ALLIED ELECTRONICS,NaN,service@suncoating.com,716-924-3066,NaN,3775,vendor
5457,ALLIED ELECTRONICS INC,1NNN2,NaN,NaN,NaN,3775,manufacturer
5456,"ALLIED ELECTRONICS, INC",1FV21,NaN,NaN,NaN,3775,manufacturer
5458,"ALLIED ELECTRONICS, INC",1QFU4,NaN,NaN,NaN,3775,manufacturer
5445,AMERICAN ZETTLER INC,26806,lance@csgparts.com,NaN,NaN,3775,manufacturer
5454,"ARAMCO, INC. DIV ARAMCO OUTSIDE BLINDS",0B0K1,NaN,NaN,NaN,3775,manufacturer
5463,BEI NORTH AMERICA LLC,7LE18,NaN,NaN,NaN,3775,manufacturer
5449,"BEI SENSORS & SYSTEMS COMPANY, INC. DBA BEI DU...",11534,sales@eltime.co.uk,NaN,NaN,3775,manufacturer
5448,BELDEN WIRE & CABLE COMPANY LLC DBA BELDEN CDT...,16428,sales@ebyco.com,NaN,NaN,3775,manufacturer
5447,BROOKS ELECTRONICS INC,63237,roy@aquaairind.com,NaN,NaN,3775,manufacturer


In [27]:
rejected_all = pd.concat(rejected, ignore_index=True)
rejected_all["reason"].value_counts().to_frame("rows")

,rows
reason,
same firm as its vendor,1678
orphan manufacturer already listed,755
duplicate vendor name,500
invalid cage code,280
orphan vendor already in linked list,231
cage not in cages table,108
marked do-not-use / inactive,104
missing vendor name,24
duplicate cage,16


## 12. Save

In [28]:
linked.to_csv(f"{OUT}/linked_vendor_manufacturers.csv", index=False)
orphan_vendors.to_csv(f"{OUT}/orphan_vendors.csv", index=False)
orphan_manufacturers.to_csv(f"{OUT}/orphan_manufacturers.csv", index=False)
rejected_all.to_csv(f"{OUT}/rejected.csv", index=False)
print("saved to", OUT)

saved to output
